# Diabetes Prediction — End-to-End Machine Learning Project

## Research question
Can we predict whether a person is diagnosed with diabetes from demographic, lifestyle, cardiovascular and metabolic features?

## Target
`diagnosed_diabetes`

- 1 = diagnosed with diabetes
- 0 = not diagnosed

## Important methodological decisions

1. `diabetes_stage` is excluded from X because it contains information that directly reveals the diagnosis and therefore creates target leakage.
2. Rows where `systolic_bp <= diastolic_bp` are removed because this is an explicit physiologically invalid relationship in the supplied dataset.
3. The target is not used for clustering.
4. The test set is kept untouched until final evaluation.
5. All learned preprocessing is fitted inside pipelines on training data only.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, silhouette_score
)

from sklearn.cluster import KMeans, AgglomerativeClustering

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Load the supplied dataset

In [ ]:
df = pd.read_csv("diabetes_dataset.csv")

print("Shape:", df.shape)
display(df.head())

## 2. Initial inspection

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

display(missing)

## 3. Target distribution

The supplied dataset is already approximately 60/40:

- diagnosed = 1: about 60%
- not diagnosed = 0: about 40%

This is not an extreme class imbalance, but stratification is still used so the train/test target proportions remain representative.

In [ ]:
target = "diagnosed_diabetes"

display(
    pd.DataFrame({
        "count": df[target].value_counts().sort_index(),
        "percentage": (df[target].value_counts(normalize=True).sort_index() * 100).round(2)
    })
)

plt.figure(figsize=(6,4))
sns.countplot(data=df, x=target)
plt.title("Target Distribution")
plt.xlabel("Diagnosed Diabetes")
plt.ylabel("Count")
plt.show()

## 4. Data-quality check: blood pressure

The supplied data contains 154 rows where:

`systolic_bp <= diastolic_bp`

These rows are removed because systolic pressure should be higher than diastolic pressure. This removes only 0.154% of the 100,000 observations.

In [ ]:
invalid_bp = df["systolic_bp"] <= df["diastolic_bp"]

print("Invalid BP rows:", invalid_bp.sum())
print("Percentage:", round(invalid_bp.mean() * 100, 3), "%")

df = df.loc[~invalid_bp].copy()

print("Shape after BP cleaning:", df.shape)

## 5. Leakage check

`diabetes_stage` is excluded deliberately.

It is not just another useful feature: values such as `No Diabetes` and `Type 2` reveal the target almost directly. Including it would let a model exploit information that would not represent an independent predictive signal.

This is **target leakage**, and a model trained with it could achieve misleadingly high performance.

In [ ]:
print(pd.crosstab(
    df["diabetes_stage"],
    df["diagnosed_diabetes"],
    normalize="index"
).round(3))

leakage_cols = ["diabetes_stage"]

X = df.drop(columns=[target] + leakage_cols)
y = df[target].astype(int)

print("Excluded:", leakage_cols)
print("X shape:", X.shape)

## 6. EDA — numerical distributions

In [ ]:
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()
categorical_cols = X.select_dtypes(include="object").columns.tolist()

display(
    X[numeric_cols].describe().T.assign(
        skewness=X[numeric_cols].skew()
    ).sort_values("skewness", key=np.abs, ascending=False)
)

In [ ]:
eda_numeric = [
    "age", "physical_activity_minutes_per_week", "diet_score",
    "sleep_hours_per_day", "screen_time_hours_per_day",
    "bmi", "waist_to_hip_ratio", "systolic_bp", "diastolic_bp",
    "heart_rate", "cholesterol_total", "hdl_cholesterol",
    "ldl_cholesterol", "triglycerides", "glucose_fasting",
    "glucose_postprandial", "insulin_level", "hba1c",
    "diabetes_risk_score"
]
eda_numeric = [c for c in eda_numeric if c in X.columns]

fig, axes = plt.subplots(4, 5, figsize=(20,14))
axes = axes.flatten()

for ax, col in zip(axes, eda_numeric):
    sns.histplot(data=df, x=col, hue=target, bins=25, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(col)

for ax in axes[len(eda_numeric):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. EDA — target relationships

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20,14))
axes = axes.flatten()

for ax, col in zip(axes, eda_numeric):
    sns.boxplot(data=df, x=target, y=col, ax=ax)
    ax.set_title(f"{col} vs target")

for ax in axes[len(eda_numeric):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. EDA — categorical variables

In [ ]:
for col in categorical_cols:
    summary = df.groupby(col)[target].agg(["mean", "count"])
    summary["diagnosed_pct"] = summary["mean"] * 100
    print(f"\n{col}")
    display(summary.sort_values("diagnosed_pct", ascending=False).round(3))

## 9. Correlation with the target

Correlation is used as an exploratory tool, not as proof of causation.

In [ ]:
corr_cols = eda_numeric + [target]
corr = df[corr_cols].corr()

plt.figure(figsize=(14,11))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Numerical Correlation Matrix")
plt.tight_layout()
plt.show()

display(
    corr[target].drop(target)
    .sort_values(key=np.abs, ascending=False)
    .to_frame("correlation")
)

## 10. EDA findings to report

After running the cells above, report at least five evidence-based observations.

Good questions include:

1. Which metabolic measurements differ most between diagnosed and non-diagnosed groups?
2. Is HbA1c associated with the target?
3. Is fasting glucose associated with the target?
4. Does BMI show a meaningful group difference?
5. How do family history and hypertension relate to diagnosis?
6. Are lifestyle variables such as activity, diet and sleep associated with the target?
7. Which variables have unusual skewness or outliers?

Do not claim causality from these plots.

# 11. Train/Test Split

The final test set is held out before model selection.

`stratify=y` preserves the approximately 60/40 class distribution in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Train target rate:", round(y_train.mean(), 4))
print("Test target rate :", round(y_test.mean(), 4))

# 12. Preprocessing

Numerical features are median-imputed and standardized.

Categorical features are most-frequent-imputed and one-hot encoded.

The transformations live inside a Pipeline so that they are fitted using training data only.

In [ ]:
numeric_features = X_train.select_dtypes(exclude="object").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))

# 13. Baseline — Logistic Regression

Because the target is binary, Logistic Regression is an appropriate interpretable baseline.

In [ ]:
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

baseline.fit(X_train, y_train)

base_prob = baseline.predict_proba(X_test)[:, 1]
base_pred = (base_prob >= 0.5).astype(int)

baseline_scores = {
    "Accuracy": accuracy_score(y_test, base_pred),
    "Precision": precision_score(y_test, base_pred),
    "Recall": recall_score(y_test, base_pred),
    "F1": f1_score(y_test, base_pred),
    "ROC-AUC": roc_auc_score(y_test, base_prob)
}

display(pd.Series(baseline_scores).to_frame("Logistic Regression"))

# 14. Compare multiple algorithms

We use:

- Logistic Regression
- KNN
- Decision Tree
- Random Forest
- Gradient Boosting

The dataset has 100k observations, so tree ensembles are especially practical candidates. KNN is included as a distance-based comparison.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),

    "KNN": Pipeline([
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=15, weights="distance"))
    ]),

    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=20,
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=250,
            max_depth=12,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("prep", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            random_state=RANDOM_STATE
        ))
    ])
}

# 15. Cross-validation

We use stratified 5-fold CV on the training set.

Primary metric: **ROC-AUC**.

Additional metrics:

- Accuracy
- Precision
- Recall
- F1

Since the classes are reasonably balanced, we can report all of them, rather than relying on Accuracy alone.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train, y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    rows.append({
        "Model": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    })

cv_results = pd.DataFrame(rows).sort_values("CV ROC-AUC", ascending=False)
display(cv_results.round(4))

# 16. Overfitting / Underfitting

A large difference between Train ROC-AUC and CV ROC-AUC suggests overfitting.

If both are poor, the model may be underfitting.

In [ ]:
cv_results["ROC-AUC Gap"] = (
    cv_results["Train ROC-AUC"] - cv_results["CV ROC-AUC"]
)

display(
    cv_results[
        ["Model", "Train ROC-AUC", "CV ROC-AUC", "ROC-AUC Gap",
         "CV Recall", "CV F1"]
    ].round(4)
)

# 17. Hyperparameter tuning — Gradient Boosting

We tune the parameters that control model complexity and learning:

- number of trees
- learning rate
- tree depth
- subsampling

A smaller learning rate with more estimators can learn gradually; deeper trees increase complexity and may overfit.

In [ ]:
gb_grid = GridSearchCV(
    models["Gradient Boosting"],
    {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
        "model__subsample": [0.8, 1.0]
    },
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True
)

gb_grid.fit(X_train, y_train)

print("Best parameters:")
print(gb_grid.best_params_)
print("Best CV ROC-AUC:", round(gb_grid.best_score_, 4))

display(
    pd.DataFrame(gb_grid.cv_results_)
    .sort_values("mean_test_score", ascending=False)
    [["param_model__n_estimators",
      "param_model__learning_rate",
      "param_model__max_depth",
      "param_model__subsample",
      "mean_train_score",
      "mean_test_score",
      "std_test_score"]]
    .head(15)
)

# 18. Hyperparameter tuning — Random Forest

In [ ]:
rf_grid = GridSearchCV(
    models["Random Forest"],
    {
        "model__n_estimators": [200, 300],
        "model__max_depth": [8, 12, 16, None],
        "model__min_samples_leaf": [2, 5, 10]
    },
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True
)

rf_grid.fit(X_train, y_train)

print("Best parameters:")
print(rf_grid.best_params_)
print("Best CV ROC-AUC:", round(rf_grid.best_score_, 4))

# 19. Final model selection

Selection is based on:

1. CV ROC-AUC
2. Recall
3. F1
4. Generalization gap
5. Practical complexity

The final test set is still untouched.

In [ ]:
candidates = {
    "Tuned Gradient Boosting": gb_grid.best_estimator_,
    "Tuned Random Forest": rf_grid.best_estimator_,
    "Logistic Regression": models["Logistic Regression"]
}

candidate_rows = []

for name, model in candidates.items():
    s = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring=scoring,
        return_train_score=True, n_jobs=-1
    )

    candidate_rows.append({
        "Model": name,
        "Train ROC-AUC": s["train_roc_auc"].mean(),
        "CV ROC-AUC": s["test_roc_auc"].mean(),
        "CV Recall": s["test_recall"].mean(),
        "CV F1": s["test_f1"].mean()
    })

candidate_results = pd.DataFrame(candidate_rows).sort_values(
    "CV ROC-AUC", ascending=False
)

display(candidate_results.round(4))

final_name = candidate_results.iloc[0]["Model"]
final_model = candidates[final_name]

print("Selected:", final_name)

# 20. Final test evaluation

In [ ]:
final_model.fit(X_train, y_train)

y_prob = final_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

final_scores = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

display(pd.Series(final_scores).to_frame("Final Test Score"))

print(classification_report(
    y_test, y_pred,
    target_names=["Not diagnosed", "Diagnosed"],
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"]
)
plt.title(f"Confusion Matrix — {final_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("TN:", cm[0,0])
print("FP:", cm[0,1])
print("FN:", cm[1,0])
print("TP:", cm[1,1])

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC Curve — {final_name}")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title(f"Precision-Recall Curve — {final_name}")
plt.show()

# 21. Threshold analysis

The default threshold is 0.50.

If the business/health objective prioritizes catching as many true diabetes cases as possible, Recall may be prioritized. Lowering the threshold usually increases Recall and decreases Precision.

We inspect the trade-off rather than assuming 0.50 is optimal.

In [ ]:
threshold_rows = []

for t in np.arange(0.20, 0.81, 0.05):
    p = (y_prob >= t).astype(int)
    threshold_rows.append({
        "Threshold": t,
        "Precision": precision_score(y_test, p, zero_division=0),
        "Recall": recall_score(y_test, p, zero_division=0),
        "F1": f1_score(y_test, p, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))

plt.figure(figsize=(9,5))
for metric in ["Precision", "Recall", "F1"]:
    plt.plot(threshold_df["Threshold"], threshold_df[metric], marker="o", label=metric)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Trade-off")
plt.legend()
plt.show()

# 22. Error analysis

For a screening-style use case, **False Negatives** deserve particular attention:

> The person is diagnosed with diabetes, but the model predicts 0.

False positives also matter because they can lead to unnecessary follow-up.

We inspect both groups.

In [ ]:
errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = y_pred
errors["probability"] = y_prob

fn = errors[(errors.actual == 1) & (errors.predicted == 0)]
fp = errors[(errors.actual == 0) & (errors.predicted == 1)]

print("False negatives:", len(fn))
print("False positives:", len(fp))

display(fn.head(10))
display(fp.head(10))

# 23. Feature importance

For tree-based Gradient Boosting, we inspect the transformed feature importance.

Because categorical variables were one-hot encoded, the resulting names correspond to encoded categories.

In [ ]:
gb_final = gb_grid.best_estimator_
gb_model = gb_final.named_steps["model"]

feature_names = gb_final.named_steps["prep"].get_feature_names_out()

importance = pd.Series(
    gb_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(importance.head(20).to_frame("importance"))

plt.figure(figsize=(10,7))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top Gradient Boosting Feature Importances")
plt.xlabel("Importance")
plt.show()

# 24. Unsupervised Learning — K-Means

Clustering must be independent of the target.

Therefore `diagnosed_diabetes` and `diabetes_stage` are NOT used.

We cluster using selected demographic/lifestyle/clinical measurements.

In [ ]:
cluster_features = [
    "age", "physical_activity_minutes_per_week", "diet_score",
    "sleep_hours_per_day", "screen_time_hours_per_day",
    "bmi", "waist_to_hip_ratio", "systolic_bp", "diastolic_bp",
    "heart_rate", "cholesterol_total", "hdl_cholesterol",
    "ldl_cholesterol", "triglycerides", "glucose_fasting",
    "glucose_postprandial", "insulin_level", "hba1c"
]

cluster_features = [c for c in cluster_features if c in df.columns]

cluster_imputer = SimpleImputer(strategy="median")
cluster_scaler = StandardScaler()

cluster_values = cluster_imputer.fit_transform(df[cluster_features])
cluster_scaled = cluster_scaler.fit_transform(cluster_values)

print(cluster_features)

In [ ]:
k_scores = []

for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(cluster_scaled)

    k_scores.append({
        "K": k,
        "Silhouette": silhouette_score(cluster_scaled, labels),
        "Inertia": km.inertia_
    })

k_df = pd.DataFrame(k_scores)
display(k_df.round(4))

plt.figure(figsize=(8,5))
plt.plot(k_df.K, k_df.Silhouette, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette Score")
plt.title("K-Means K Selection")
plt.show()

In [ ]:
best_k = int(k_df.loc[k_df.Silhouette.idxmax(), "K"])

kmeans = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=RANDOM_STATE
)

df["KMeans_Cluster"] = kmeans.fit_predict(cluster_scaled)

print("Best K:", best_k)
print("Silhouette:", round(
    silhouette_score(cluster_scaled, df["KMeans_Cluster"]), 4
))

display(df["KMeans_Cluster"].value_counts().sort_index().to_frame("size"))

## 25. Cluster profiling

The target is used **only after clustering** to describe whether discovered groups have different diabetes rates. It is not used to create the clusters.

In [ ]:
cluster_profile = df.groupby("KMeans_Cluster")[cluster_features].mean().round(2)
cluster_profile["count"] = df["KMeans_Cluster"].value_counts().sort_index()

display(cluster_profile)

cluster_target = df.groupby("KMeans_Cluster")[target].agg(["mean", "count"])
cluster_target["diagnosed_pct"] = cluster_target["mean"] * 100

display(cluster_target.round(3))

# 26. Second clustering algorithm — Agglomerative

We compare K-Means with Agglomerative Clustering using the same scaled features and K.

In [ ]:
agg = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
agg_labels = agg.fit_predict(cluster_scaled)

agg_score = silhouette_score(cluster_scaled, agg_labels)

df["Agg_Cluster"] = agg_labels

print("Agglomerative Silhouette:", round(agg_score, 4))
display(df["Agg_Cluster"].value_counts().sort_index().to_frame("size"))

In [ ]:
agg_profile = df.groupby("Agg_Cluster")[cluster_features].mean().round(2)
agg_profile["count"] = df["Agg_Cluster"].value_counts().sort_index()

display(agg_profile)

agg_target = df.groupby("Agg_Cluster")[target].mean().mul(100).round(2)
display(agg_target.to_frame("diagnosed_pct"))

# 27. Final comparison table

The final report should clearly distinguish:

- Cross-validation results used for model selection
- Final test results used for the final unbiased estimate
- Clustering results, which are a separate unsupervised analysis

In [ ]:
final_table = pd.DataFrame([final_scores], index=[final_name])
display(final_table.round(4))

print("K-Means silhouette:", round(
    silhouette_score(cluster_scaled, df["KMeans_Cluster"]), 4
))
print("Agglomerative silhouette:", round(agg_score, 4))

# 28. Final conclusions

Use the actual numbers produced by the notebook.

### Recommended structure

**Data quality**
- 100,000 original observations.
- 154 invalid blood-pressure rows removed.
- No missing values were observed in the supplied file.
- `diabetes_stage` excluded because of target leakage.

**EDA**
- Discuss at least five observed relationships.
- Prioritize metabolic variables, cardiovascular variables and lifestyle factors.
- Avoid causal claims.

**Modeling**
- Compare Logistic Regression, KNN, Decision Tree, Random Forest and Gradient Boosting.
- Use cross-validation.
- Discuss train/validation gaps.

**Final model**
- Report test Accuracy, Precision, Recall, F1 and ROC-AUC.
- Explain why the final model was selected.

**Error analysis**
- Discuss false negatives and false positives.
- Explain threshold trade-offs.

**Clustering**
- Compare K-Means and Agglomerative Clustering using Silhouette Score and cluster profiles.
- Explain what the clusters represent.

**Limitations**
- The dataset is synthetic/curated and should not be treated as clinical validation unless the dataset documentation confirms otherwise.
- High predictive performance does not prove clinical usefulness.
- Association does not imply causation.
- A real deployment would require external validation and clinical governance.